## Blood Pressure

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
os.chdir(r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\E_ResearchData\2_ResearchData\20240822')

In [ ]:
metingen = pd.read_sas('meting_20240822.sas7bdat', encoding='latin1')
metingen.studyId_0831 = metingen.studyId_0831.astype('uint64')

In [ ]:
df_consult = pd.read_sas(
    r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\E_ResearchData\2_ResearchData\20240822\consult_20240822.sas7bdat',
    encoding='latin1')
df_consult.studyId_0831 = df_consult.studyId_0831.astype('uint64')

In [ ]:
df_consult = df_consult.drop(['Consult_id', 'ConsultStatus', 'type_code_original',
                              'type_display_original', 'created', 'title', 'author_usercode',
                              'initial_author_usercode', 'custodian_Organization_value',
                              'SpecialismeNaam', 'initial_custodian_Organization_v',
                              'SpecialismeNaamInitial', 'LocationCode', 'LocationName',
                              'PatientProtected', 'section_title_code_original',
                              'section_title_display_original', 'section_text',
                              'section_text_contentType', 'orderedBy', 'index_date'], axis='columns')

In [ ]:
df_consult = df_consult.groupby('studyId_0831').last()
df_consult

In [ ]:
metingen.columns

In [ ]:
metingen = metingen.drop(['identifier_value', 'category_display_original',
                          'LOINC', 'code_code_original','code_display_original',
                          'code_display2_original', 'comment', 'context_Encounter',
                          'context_Encounter_system', 'context_Encounter_value',
                          'source', 'sourcetype', 'status', 'index_date'],
                          axis='columns')
metingen

In [ ]:
metingen[['DiastolicBP', 'SystolicBP', 'MeanBP']].plot(kind='box',
                                                       subplots=True,
                                                              figsize=[15, 8])

In [ ]:
metingen.studyId_0831.value_counts()

In [ ]:
metingen_filtered = metingen.query('DiastolicBP >= 50 and DiastolicBP <= 250 and '\
                                   'SystolicBP >= 50 and SystolicBP <= 250 and '
                                   'MeanBP >= 50 and MeanBP <= 250')
metingen_filtered

In [ ]:
metingen_filtered[['DiastolicBP','SystolicBP','MeanBP']].plot(kind='box',
                                                              subplots=True,
                                                              figsize=[12,8])

In [ ]:
metingen_filtered.studyId_0831.value_counts()

In [ ]:
df_merged = pd.merge(metingen_filtered, df_consult, right_index=True,
                     left_on='studyId_0831', how='left')
df_merged

In [ ]:
df_merged['delta'] = (df_merged.date - df_merged.effectiveDateTime).dt.days
df_merged

In [ ]:
df_merged.studyId_0831.value_counts()

In [ ]:
df_merged.delta.hist(bins=100)

In [ ]:
df_merged = df_merged.query('delta < 28 and delta > -28').reset_index(drop=True)
df_merged

In [ ]:
df_merged.studyId_0831.nunique()

In [ ]:
unique_before = set(df_merged.query('delta <= 0').studyId_0831.unique())
unique_after = set(df_merged.query('delta >= 0').studyId_0831.unique())

print(len(unique_before.intersection(unique_after)))
print(len(unique_before), len(unique_after),)

In [ ]:
df_merged.studyId_0831.value_counts()

In [ ]:
df_bp = df_merged[['studyId_0831', 'DiastolicBP','SystolicBP','MeanBP']]
df_bp.studyId_0831.value_counts()

In [ ]:
timeseries= df_merged[['studyId_0831', 'DiastolicBP', 'SystolicBP', 'MeanBP']]

In [ ]:
def weighted_avg(x: pd.Series):
    deltas = df_merged.loc[x.index,'delta']
    if x.count() == 0:
        return np.nan
    else:
        w = 1/np.abs(deltas.to_numpy() + 1e-15)
        w = w/w.sum()
        return np.average(x.loc[x.notna()].to_numpy(),weights=w)
        

In [ ]:
df_bp = df_bp.groupby('studyId_0831').agg(weighted_avg)
df_bp

In [ ]:
df_bp.notna().sum()

In [ ]:
df_bp.plot(kind='box',figsize=[8,6])

---

In [ ]:
import statsmodels.api as sm

In [ ]:
spector_data = sm.datasets.spector.load()

In [ ]:
y = timeseries.query('studyId_0831 == 58866825127')['DiastolicBP']

X = np.arange(len(y))
X = sm.add_constant(X)
#timeseries['DiastolicBP'] = sm.add_constant(timeseries['DiastolicBP'], prepend=False)
mod = sm.OLS(y,X)

In [ ]:
res = mod.fit()
print(res.summary())

In [ ]:
res.params['x1']

In [ ]:
plt.plot(X[:,1],res.params['const'] + res.params['x1']*X[:,1])
plt.plot(X[:,1],y)